In [ ]:
# Environment: harpy

# Import packages
import sparrow as sp
import os
import numpy as np
import spatialdata as sd
from sparrow.table import add_table_layer
from skimage import io, img_as_ubyte
from scipy.sparse import csr_matrix
from sklearn.neighbors import BallTree
from joblib import Parallel, delayed

# Specify paths
sdata_path = r'd:\Data\2024-07-UTBenchmark-SpC\data\processed\BMK\exp74\sdata.zarr'
output_path = r'd:\Data\2024-07-UTBenchmark-SpC\data\processed\BMK\exp74'

# Read sdata
sdata = sd.read_zarr(sdata_path)

# Define hires_img
hires_img = sdata['raw_image'].compute().data

def sliding_window_binning(adata, radius, batch_size=1000):
    coordinates = adata.obsm['spatial']
    expression_data = adata.X
    if not isinstance(expression_data, csr_matrix):
        expression_data = csr_matrix(expression_data)

    tree = BallTree(coordinates)
    n_batches = int(np.ceil(coordinates.shape[0] / batch_size))

    def aggregate_batch(start_idx, end_idx):
        results = []
        for i in range(start_idx, end_idx):
            nearby_spots = tree.query_radius([coordinates[i]], r=radius)[0]
            aggregated_expression = expression_data[nearby_spots].sum(axis=0)
            results.append(aggregated_expression)
        return results

    aggregated_expression = []
    for i in range(n_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, coordinates.shape[0])
        aggregated_expression.extend(aggregate_batch(start_idx, end_idx))
    
    aggregated_expression = csr_matrix(np.vstack(aggregated_expression))
    new_adata = adata.copy()
    new_adata.X = aggregated_expression

    return new_adata

# Calculate sliding window expression values
level_name = 'L13_heAuto'
new_adata = sliding_window_binning(sdata.tables[f'table_{level_name}'], radius=20)
sdata = add_table_layer(sdata, adata=new_adata.copy(), output_layer=f'table_{level_name}_sliding_window', region=[f"labels_spots_{level_name}"], overwrite=True)

# Leiden clustering
import gc # garbage collection to avoid out-of-memory issues.

for table_name in [level_name, f'{level_name}_sliding_window']:
    # Preprocess data
    sdata = sp.tb.preprocess_transcriptomics(
        sdata=sdata,
        size_norm=False, # scanpy.sc.pp.normalize_total is used for normalization
        labels_layer=f"labels_spots_{level_name}",
        table_layer=f"table_{table_name}",
        output_layer=f"table_{table_name}_processed",
        min_counts=50,
        min_cells=5,
        highly_variable_genes=False,
        max_value_scale=10,
        n_comps = 50,
        update_shapes_layers=False,
        overwrite=True
    )
    
    sp.pl.preprocess_transcriptomics(
        sdata, 
        table_layer=f"table_{table_name}_processed",
        output=os.path.join(output_path, f"{table_name}_preprocess")
    )

    gc.collect()

    # Leiden clustering
    leiden_resolutions = [0.6]

    for res in leiden_resolutions:
        res_str = str(res).replace('.', '')
        
        sdata = sp.tb.leiden(
            sdata, 
            labels_layer=f"labels_spots_{level_name}",
            table_layer=f"table_{table_name}_processed",
            output_layer=f"table_{table_name}_processed",
            calculate_umap=True,
            calculate_neighbors=True,
            rank_genes=True,
            n_neighbors=35,
            n_pcs=30,
            resolution=res,
            key_added=f'leiden_{res_str}',
            index_names_var=None,
            index_positions_var=None,
            random_state=100,
            overwrite=True
        )

        sp.pl.cluster(
            sdata, 
            table_layer=f"table_{table_name}_processed",
            key_added=f'leiden_{res_str}',
            output=os.path.join(output_path, f'UMAP_{table_name}_{res_str}')
        )

    gc.collect()

    # Save raw adata with H&E added
    adata = sdata.tables[f'table_{table_name}'].copy()

    library_id = level_name
    adata.uns["spatial"] = {library_id: {}}
    adata.uns["spatial"][library_id]["images"] = {}
    adata.uns['spatial'][library_id]['images']['hires'] = img_as_ubyte(hires_img.transpose(1, 2, 0))
    adata.uns['spatial'][library_id]['use_quality'] = 'hires' 
    adata.uns['spatial'][library_id]['scalefactors'] = {"tissue_hires_scalef": 1}

    adata.write(f'{output_path}/adata_{table_name}.h5ad')
    
    # Save processed adata with H&E added
    adata = sdata.tables[f'table_{table_name}_processed'].copy()

    library_id = level_name
    adata.uns["spatial"] = {library_id: {}}
    adata.uns["spatial"][library_id]["images"] = {}
    adata.uns['spatial'][library_id]['images']['hires'] = img_as_ubyte(hires_img.transpose(1, 2, 0))
    adata.uns['spatial'][library_id]['use_quality'] = 'hires' 
    adata.uns['spatial'][library_id]['scalefactors'] = {"tissue_hires_scalef": 1}

    adata.write(f'{output_path}/adata_{table_name}_processed.h5ad')